# 03.1 — Parsing by format

Module 02 read seven PDFs and scored 0.444. Twelve of the thirty-six questions
pointed at documents it never opened — a spreadsheet, a slide deck, an email
thread, a scan, a Word memo, an HTML page, a CSV and a markdown file.

Eight formats. Each needs a different library, and each fails in its own way.
This notebook writes a handler for every one of them and shows the failure
before fixing it.

By the end you'll have a `parse()` function that reads anything in the corpus.

In [1]:
!pip install -q pdfplumber==0.11.10 pypdf==6.17.0 pymupdf4llm==1.28.2 python-docx==1.2.0 \
                openpyxl==3.1.5 python-pptx==1.0.2 beautifulsoup4==4.15.0 lxml==6.1.3

In [2]:
from pathlib import Path

CORPUS = Path('../../corpus/docs')

for p in sorted(CORPUS.iterdir()):
    print(f'{p.suffix:<7} {p.name}')

.pdf    kaduna-agro-annual-report-2024.pdf
.pptx   kaduna-agro-board-deck-2025-01.pptx
.pdf    kaduna-agro-board-minutes-2024-10-17.pdf
.xlsx   kaduna-agro-distribution-2024.xlsx
.pdf    kdirs-guidance-note-4-2024-SCANNED.pdf
.pdf    nfsc-circular-2024-07-cybersecurity.pdf
.pdf    nfsc-circular-2025-02-amendment.pdf
.html   nfsc-circular-2025-02.html
.csv    sahel-approved-vendors.csv
.md     sahel-branch-runbook.md
.pdf    sahel-employee-handbook-2023.pdf
.pdf    sahel-employee-handbook-2025.pdf
.docx   sahel-hr-memo-2024-41-TRACKED.docx
.eml    sahel-per-diem-thread.eml
.pdf    sahel-procurement-policy-v3.pdf


## Markdown — the control

Start here so the rest has something to be compared against.

In [3]:
def read_md(path):
    return path.read_text(encoding='utf-8')

text = read_md(CORPUS / 'sahel-branch-runbook.md')
print(f'{len(text):,} chars')
print(text[:200])

1,298 chars
# Sahel Microfinance Bank — Branch Operations Runbook

> Internal. Version 2.4. Owner: Head, Branch Operations.

## Opening procedure

1. Dual custody of the vault is mandatory. Neither key holder may


One line. Headings intact, table intact, nothing lost.

Worth noticing how much of the difficulty in the rest of this notebook is
*inherent to the format* rather than inherent to RAG. Markdown was designed to
be read by machines. A PDF was designed to be printed.

## CSV — line count is not record count

In [4]:
import csv

path = CORPUS / 'sahel-approved-vendors.csv'

with path.open(newline='', encoding='utf-8') as f:
    rows = list(csv.reader(f))

comments = [r for r in rows if r and r[0].lstrip().startswith('#')]

print('physical lines (what wc -l counts):', len(path.read_text().splitlines()))
print('rows csv.reader returns          :', len(rows))
print('actual vendor records            :', len(rows) - 1 - len(comments))

physical lines (what wc -l counts): 14
rows csv.reader returns          : 7
actual vendor records            : 5


Three different answers to "how many rows is this file", and none of them is
wrong — they're counting different things.

**14** is what `wc -l` and `readlines()` see. The addresses contain newlines
inside quoted fields, so a line is not a record.

**7** is what `csv.reader` returns once it handles the quoting properly. That
includes the header row and a `#` comment line at the end of the file, which
`csv.reader` has no reason to treat as special and hands back as data.

**5** is the number of vendors, which is what anyone asking a question about this
file actually means.

Getting from 14 to 5 takes a library that understands the format and a decision
about the comment line. Neither is hard. Both are easy to skip.

One more decision. A row of comma-separated values embeds badly — the column
names end up in a different chunk from the values, so a chunk reading
`V-0080, Etim Print Concepts, Printing, ...` gives a retriever nothing to match
against. Labelling each field keeps the two together.

In [5]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        rows = [r for r in csv.reader(f)
               if r and not r[0].lstrip().startswith('#')]
    header, *body = rows
    return '\n\n'.join(
        '\n'.join(f'{h}: {v}' for h, v in zip(header, row)) for row in body
    )

print(read_csv(path)[:300])

vendor_id: V-0041
vendor_name: Bako Logistics Limited
category: Haulage
address: 12 Ahmadu Bello Way
Kaduna South
Kaduna State
tcc_expiry: 2025-12-31
contract_value_ngn: 840000000

vendor_id: V-0058
vendor_name: Okonkwo & Sons Nigeria Ltd
category: Packaging
address: Plot 3, Trans-Amadi Industrial L


## HTML — the content is a tenth of the file

In [6]:
from bs4 import BeautifulSoup

raw = (CORPUS / 'nfsc-circular-2025-02.html').read_text(encoding='utf-8')
print(f'raw file: {len(raw):,} chars')
print('everything, tags stripped:')
print(BeautifulSoup(raw, 'lxml').get_text(' ', strip=True)[:260])

raw file: 3,542 chars
everything, tags stripped:
Circular NFSC/CIR/2025/02 | National Financial Services Commission Skip to main content National Financial Services Commission Home About the Commission Licensing Circulars Returns Portal Careers Contact Home › Circulars › 2025 › NFSC/CIR/2025/02 The Returns P


Navigation, breadcrumb, a maintenance banner, a sidebar, a footer. All of it
would end up in your chunks, and "Skip to main content" would be retrievable.

Strip the furniture and take the content element:

In [7]:
def read_html(path):
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'lxml')
    for tag in soup(['nav', 'header', 'footer', 'aside', 'script', 'style', 'form']):
        tag.decompose()
    main = soup.find('main') or soup.body
    return main.get_text('\n', strip=True)

text = read_html(CORPUS / 'nfsc-circular-2025-02.html')
print(f'{len(text):,} chars | nav still present: {"Returns Portal" in text}')
print(text[:260])

1,368 chars | nav still present: False
Circular NFSC/CIR/2025/02
Issued 3 March 2025 · Effective 1 April 2025 · Status: In force
Amendment to the Cybersecurity and Operational Resilience Framework
1. Purpose
This circular amends Circular NFSC/CIR/2024/07. Provisions of the earlier circular not
addr


From a few thousand characters to about 1,400, and the 1,400 are the circular.

`<main>` is the reliable hook when a page has one. When it doesn't, you're
guessing at class names — which is why HTML ingestion at scale is a per-site job
rather than a general one.

## Word — three approaches, three different answers

This is the one worth slowing down for.

`sahel-hr-memo-2024-41-TRACKED.docx` is the memo that produced the 2025 handbook
changes. It has **unaccepted tracked changes** in it: `NGN 35,000` deleted,
`NGN 60,000` inserted.

Here are three reasonable ways to read it.

In [8]:
import re, zipfile
import docx

path = CORPUS / 'sahel-hr-memo-2024-41-TRACKED.docx'

# 1. the obvious library call
supported = '\n'.join(p.text for p in docx.Document(path).paragraphs)

# 2. unzip it and strip the tags
xml = zipfile.ZipFile(path).read('word/document.xml').decode('utf-8', 'ignore')
stripped = re.sub(r'<[^>]+>', '', xml)

for label, text in [('python-docx', supported), ('naive xml', stripped)]:
    print(f"{label:<14} sees 35,000: {'35,000' in text!s:<6} sees 60,000: {'60,000' in text}")

i = supported.find('revised to')
print(f'\npython-docx   -> {supported[i:i + 40]!r}')
j = stripped.find('revised to')
print(f'\naive xml   -> {stripped[j:j + 60]!r}')

python-docx    sees 35,000: False  sees 60,000: False
naive xml      sees 35,000: True   sees 60,000: True

python-docx   -> 'revised to . The per diem continues to c'

aive xml   -> 'revised to NGN 35,000 per nightNGN 60,000 per night. The per'


Read that carefully.

**python-docx sees neither number.** `paragraph.text` skips `w:del` and `w:ins`
runs entirely, so the sentence arrives as *"the per diem is revised to ."* The
figure is simply gone.

**Stripping the XML sees both**, concatenated with no separator:
`"revised to NGN 35,000 per nightNGN 60,000 per night"`. The chunk now asserts
two contradictory amounts.

Neither raised an error. Neither logged a warning. One silently deleted the
answer and the other silently duplicated it.

This is the best argument in the course against "we used a supported library, so
parsing is handled." The library is fine. It made a reasonable default choice
about tracked changes that happens to be wrong for this document, and nothing
told you.

The fix is to decide explicitly: keep insertions, drop deletions.

In [9]:
from lxml import etree

W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'

def read_docx(path):
    """Resolve tracked changes: keep what was inserted, drop what was deleted."""
    root = etree.fromstring(zipfile.ZipFile(path).read('word/document.xml'))

    for deleted in root.iter(f'{W}del'):
        deleted.getparent().remove(deleted)

    paragraphs = []
    for para in root.iter(f'{W}p'):
        text = ''.join(t.text or '' for t in para.iter(f'{W}t'))
        if text.strip():
            paragraphs.append(text)
    return '\n'.join(paragraphs)

text = read_docx(path)
print(f"sees 35,000: {'35,000' in text} | sees 60,000: {'60,000' in text}")
i = text.find('revised to')
print(text[i:i + 60])

sees 35,000: False | sees 60,000: True
revised to NGN 60,000 per night. The per diem continues to c


Now the memo reads NGN 60,000, with no trace of the figure it replaced.

That's the answer we wanted, and it's worth being clear about why we wanted it:
we decided to read the document as though the changes had been accepted. Nobody
has actually accepted them.

You could argue the opposite just as well. It's still a draft, so it doesn't say
60,000 — it says *someone is proposing* to change 35,000 to 60,000. Under that
reading you'd refuse to index the file at all and flag it for a human to accept
or reject first.

Which policy fits depends on the document and who relies on it. A draft policy
circulating for comment is not the same thing as a memo everyone has already
acted on.

So the trouble with `python-docx` and the XML strip isn't that they picked the
wrong policy. It's that they picked one **silently**, and handed you a result you
never agreed to.

## Excel

In [10]:
from openpyxl import load_workbook

path = CORPUS / 'kaduna-agro-distribution-2024.xlsx'
wb = load_workbook(path)

for sheet in wb.worksheets:
    print(f'{sheet.title:<28} state={sheet.sheet_state}')

print('\nformula cell F3:', wb['Distribution 2024']['F3'].value)
print('with data_only=True:', load_workbook(path, data_only=True)['Distribution 2024']['F3'].value)

Distribution 2024            state=visible
Notes                        state=visible
Draft - DO NOT CIRCULATE     state=hidden

formula cell F3: =D3*1000/C3
with data_only=True: None


Three things there.

**A hidden sheet**, named `Draft - DO NOT CIRCULATE`, holding a superseded
pre-audit figure. Iterate over `wb.worksheets` without checking `sheet_state`
and you index a number somebody deliberately hid.

**A formula with no cached value.** Read the workbook normally and cell F3 is the
string `=D3*1000/C3` — which you'd embed as if it were content. Read it with
`data_only=True` and it's `None`, because the file was written by a library that
never calculated it. Neither is the number.

**A merged header cell** spanning two columns, so the header row doesn't line up
with the data.

None of these are exotic. Every spreadsheet a client hands you has at least one.

In [11]:
def read_xlsx(path):
    values = load_workbook(path, data_only=True)
    parts = []
    for sheet in values.worksheets:
        if sheet.sheet_state != 'visible':
            print(f'   skipping hidden sheet: {sheet.title!r}')
            continue
        parts.append(f'## {sheet.title}')
        for row in sheet.iter_rows():
            cells = [str(c.value) if c.value is not None else '' for c in row]
            if any(cells):
                parts.append(' | '.join(cells).rstrip(' |'))
    return '\n'.join(parts)

text = read_xlsx(path)
print(f"\n{len(text):,} chars | pre-audit figure leaked: {'264100' in text}")
print(text[:220])

   skipping hidden sheet: 'Draft - DO NOT CIRCULATE'

1,050 chars | pre-audit figure leaked: False
## Distribution 2024
 |  | Performance
State | LGA | Volume (t) | Revenue (NGN m) | Opened | Rev per tonne
Lagos | Ikeja | 21999 | 12969.8 | 2019-08-22 00:00:00
Lagos | Apapa | 4119 | 2193.3 | 2021-07-18 00:00:00
Ogun | 


## PowerPoint — the answer is in the notes

In [12]:
from pptx import Presentation

prs = Presentation(CORPUS / 'kaduna-agro-board-deck-2025-01.pptx')

slide = prs.slides[2]
print('ON THE SLIDE:')
for shape in slide.shapes:
    if shape.has_text_frame and shape.text_frame.text.strip():
        print(' ', shape.text_frame.text.replace('\n', ' | ')[:100])

print('\nIN THE SPEAKER NOTES:')
print(' ', slide.notes_slide.notes_text_frame.text[:160])

ON THE SLIDE:
  Ogbomoso commissioning
  Land title matters resolved December 2024 | Civil works 61% complete | Commissioning targeted H2 202

IN THE SPEAKER NOTES:
  Internal estimate of remaining capital requirement for Ogbomoso is NGN 2.4 billion, of which NGN 1.6 billion is uncommitted. This figure has not been disclosed 


The slide says three closures were approved. The notes name them.

An extractor that walks slide shapes and stops there loses every fact that lives
in the notes — and in a board deck, that's most of the interesting ones.

But look at what else is in there. Another slide's notes carry a capital figure
and say plainly that it hasn't been disclosed externally. Including notes
indiscriminately means indexing material somebody marked internal.

That's a governance decision, not a parsing one, and it belongs to whoever owns
the documents. Here we'll include them and tag them, so a later filter can act
on it.

In [13]:
def read_pptx(path):
    prs = Presentation(path)
    parts = []
    for i, slide in enumerate(prs.slides, 1):
        body = [sh.text_frame.text for sh in slide.shapes
               if sh.has_text_frame and sh.text_frame.text.strip()]
        parts.append(f'## Slide {i}\n' + '\n'.join(body))
        if slide.has_notes_slide:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                parts.append(f'[speaker notes]\n{notes}')
    return '\n\n'.join(parts)

text = read_pptx(CORPUS / 'kaduna-agro-board-deck-2025-01.pptx')
print(f"{len(text):,} chars | notes captured: {'2.4 billion' in text}\n")
print(f"Slides:\n{text[:643]}")

1,689 chars | notes captured: True

Slides:
## Slide 1
Kaduna Agro Processing Limited
Board Strategy Session — January 2025

[speaker notes]
SYNTHETIC DOCUMENT — generated for the RAG Engineering course. Fictional organisation. Not a real record.

## Slide 2
Operational performance 2024
Throughput 271,400 tonnes
Utilisation 79.8% against installed capacity
Zaria second line commissioned April 2024
Unplanned downtime reduced following boiler replacement

[speaker notes]
Prior year utilisation was 68.1%. The Zaria line ran at 104% of nameplate in August and September. Do not disclose the nameplate overrun externally — it reflects a temporary feedstock blend that is not repeatable.


## Email — the same content, three times

In [14]:
from email import policy
from email.parser import BytesParser

path = CORPUS / 'sahel-per-diem-thread.eml'
msg = BytesParser(policy=policy.default).parse(path.open('rb'))
body = msg.get_body(preferencelist=('plain',)).get_content()

print(f'{len(body.splitlines())} lines total')
print(f"{sum(1 for l in body.splitlines() if l.lstrip().startswith('>'))} of them quoted")
print()
print(body[:400])

45 lines total
29 of them quoted

Ibrahim,

Confirmed. Finance will apply the revised per diem to journeys commencing on or after
1 January 2025. Journeys already underway on that date are paid at the old rate for the
whole trip — we are not pro-rating.

On the corps members question: they are not covered. The memo is silent because the
welfare review did not consider them. I have asked HR to raise it separately.

Regards,
Grace E


A three-deep reply chain. The original announcement is quoted twice, so naive
chunking indexes near-identical text three times and retrieval returns three
copies of one fact.

The answers to both questions in this thread are in the top message. Everything
below is noise that looks like signal.

Signature blocks are the other problem — addresses and phone numbers diluting
every chunk they land in.

In [15]:
def read_eml(path):
    msg = BytesParser(policy=policy.default).parse(path.open('rb'))
    body = msg.get_body(preferencelist=('plain',)).get_content()

    kept = []
    for line in body.splitlines():
        if line.lstrip().startswith('>'):
            break                      # everything below is quoted history
        kept.append(line)

    header = (f"From: {msg['From']}\nTo: {msg['To']}\n"
             f"Date: {msg['Date']}\nSubject: {msg['Subject']}")
    return header + '\n\n' + '\n'.join(kept).strip()

text = read_eml(path)
print(f'{len(text):,} chars')
print(text[:340])

716 chars
From: Grace Etim <g.etim@sahelmfb.example>
To: Ibrahim Sani <i.sani@sahelmfb.example>
Date: Mon, 18 Nov 2024 16:47:11 +0100
Subject: RE: RE: Per diem revision - implementation questions

Ibrahim,

Confirmed. Finance will apply the revised per diem to journeys commencing on or after
1 January 2025. Journeys already underway on that date ar


Breaking at the first quoted line is crude and works for this thread. It will
fail on top-posted replies, on clients that don't use `>`, and on anything
forwarded. Robust email parsing is a genuinely hard problem with libraries
devoted to it.

Worth keeping the header, though. Sender, date and subject are metadata you'll
want in module 5 of this notebook series, and they're free here.

## The scan

One document left, and it doesn't belong in this notebook.

`kdirs-guidance-note-4-2024-SCANNED.pdf` has a `.pdf` extension and no text in
it at all — it's an image of a photocopy. Every function above would return an
empty string, cheerfully.

Detecting that case, and deciding what to do about it, is notebook 3.

## PDF — the choice we never justified

We've been using `pymupdf4llm` since module 02 because I told you to. PDF has more
parser options than any other format here, and it's eight of our fifteen files, so
it deserves the same treatment as everything else in this notebook.

Four common choices, on the hardest document we have.

In [16]:
import pymupdf, pymupdf4llm, pdfplumber
from pypdf import PdfReader

path = CORPUS / 'kaduna-agro-annual-report-2024.pdf'

out = {}
out['pymupdf4llm'] = pymupdf4llm.to_markdown(str(path))

with pymupdf.open(path) as doc:
    out['pymupdf raw'] = '\n'.join(page.get_text() for page in doc)

out['pypdf'] = '\n'.join(p.extract_text() or '' for p in PdfReader(path).pages)

with pdfplumber.open(path) as pdf:
    out['pdfplumber'] = '\n'.join(p.extract_text() or '' for p in pdf.pages)

for name, text in out.items():
    print(f'{name:<14} {len(text):>7,} chars')

pymupdf4llm      6,120 chars
pymupdf raw      5,846 chars
pypdf            5,880 chars
pdfplumber       5,840 chars


Similar sizes. Now look at what each one did to the distribution table.

In [17]:
for name, text in out.items():
    i = text.find('Ikeja')
    print(f'[{name}]')
    print(' ', repr(text[i - 25:i + 85]) if i > 0 else 'not found')
    print()

[pymupdf4llm]
  '|---|---|---|---|\n|Lagos|Ikeja|31,447|18,380.5|95%|\n|Lagos|Apapa|32,316|16,347.0|95%|\n|Lagos|Ikorodu|40,294|18'

[pymupdf raw]
  'NGN m)\nUtilisation\nLagos\nIkeja\n31,447\n18,380.5\n95%\nLagos\nApapa\n32,316\n16,347.0\n95%\nLagos\nIkorodu\n40,294\n18,377'

[pypdf]
  'NGN\tm)\nUtilisation\nLagos\nIkeja\n31,447\n18,380.5\n95%\nLagos\nApapa\n32,316\n16,347.0\n95%\nLagos\nIkorodu\n40,294\n18,377'

[pdfplumber]
  'NGN m) Utilisation\nLagos Ikeja 31,447 18,380.5 95%\nLagos Apapa 32,316 16,347.0 95%\nLagos Ikorodu 40,294 18,377'



Three of the four destroyed the table.

`pymupdf raw` and `pypdf` put every cell on its own line — `Lagos`, newline,
`Ikeja`, newline, `31,447`. Nothing says those three belong together, and a chunk
of that is a column of orphaned values.

`pdfplumber`'s text extraction keeps cells on one line separated by spaces, which
is better, but there's still nothing marking it as a table.

`pymupdf4llm` is the only one that emits markdown pipes. That's why we've been
using it: the structure survives into the chunk, and an LLM reading
`|Lagos|Ikeja|31,447|` can at least tell those are related.

But there's a fifth option that isn't text extraction at all.

In [18]:
with pdfplumber.open(path) as pdf:
    for pn, page in enumerate(pdf.pages, 1):
        for table in page.extract_tables():
            print(f'page {pn}: {len(table)} rows x {len(table[0])} cols')
            print(f' header row: {table[0]}')
            print()

page 1: 8 rows x 5 cols
 header row: ['State', 'Local Government Area', 'Volume (tonnes)', 'Revenue (NGN m)', 'Utilisation']

page 2: 41 rows x 5 cols
 header row: ['', '', '', '', '']

page 2: 2 rows x 1 cols
 header row: ["Ogun Ijebu-Ode 43,379 18,564.4 66%\nOyo Ibadan North 31,487 17,136.7 80%\nOyo Ogbomoso 44,403 19,978.0 41%\nOyo Oyo Town 36,427 15,736.5 43%\nKaduna Kaduna South 14,265 8,375.2 79%\nKaduna Zaria 3,771 2,141.3 61%\nKaduna Kafanchan 30,670 16,325.1 53%\nKano Kano Municipal 35,821 16,635.0 59%\nKano Dala 34,553 14,542.4 46%\nKano Wudil 31,771 17,296.5 67%\nRivers Port Harcourt 37,927 23,104.4 94%\nRivers Obio-Akpor 7,252 4,021.1 61%\nRivers Bonny 16,851 8,719.5 42%\nDelta Warri 6,402 3,373.8 47%\nDelta Asaba 28,040 12,351.1 59%\nDelta Ughelli 27,130 11,739.1 42%\nEnugu Enugu North 46,685 19,612.5 54%\nEnugu Nsukka 5,229 2,663.1 86%\nAnambra Onitsha 27,845 13,915.8 77%\nAnambra Awka 43,051 19,705.0 84%\nAnambra Nnewi 19,480 9,428.5 60%\nKwara Ilorin West 23,596 9,978.2

`pdfplumber.extract_tables()` doesn't extract text — it detects table structure and
hands back rows and columns.

On page 1 it does something no other parser managed: it returns the real header
row, `['State', 'Local Government Area', 'Volume (tonnes)', ...]`, attached to the
data.

On page 2 it falls apart. The table continues across the page break, and pdfplumber
returns 41 rows with an **empty** header, plus a second bogus table that has
swallowed the rest of the page — the directors' responsibilities section and all —
into a single cell.

So the scoreboard on this one table is:

| Parser | Table structure | Headers |
| --- | --- | --- |
| pypdf | none | none |
| pymupdf raw | none | none |
| pdfplumber (text) | none | none |
| pymupdf4llm | markdown pipes | promotes a data row to header at the page break |
| pdfplumber (`extract_tables`) | real rows and columns | correct on page 1, empty on page 2 |

**No parser gets it right.** They fail in different places, which is the actual
lesson: parser choice isn't about finding the good one, it's about knowing which
failure you've signed up for.

We keep `pymupdf4llm` because our corpus is mostly prose and markdown structure
survives chunking better than nothing does. On a corpus that was mostly financial
tables, the right answer would be a separate `extract_tables()` path — and stitching
the page-break fragments back together by hand, which is exactly what module 15
does.

Two things you're not seeing here that matter later. `pymupdf4llm` silently ran OCR
on the chart image in this document — that's the garbled text you'd find if you
scrolled through its output, and nothing in the result distinguishes it from real
text. And none of these five will read the scanned document at all.

## The dispatcher

Seven handlers. Something has to choose between them.

In [19]:
import pymupdf4llm

def read_pdf(path):
    return pymupdf4llm.to_markdown(str(path))

HANDLERS = {
    '.pdf': read_pdf,
    '.docx': read_docx,
    '.xlsx': read_xlsx,
    '.pptx': read_pptx,
    '.html': read_html,
    '.eml': read_eml,
    '.csv': read_csv,
    '.md': read_md,
}

def parse(path):
    path = Path(path)
    handler = HANDLERS.get(path.suffix.lower())
    if handler is None:
        return ValueError(f'no handler for {path.suffix}')
    return handler(path)

for p in sorted(CORPUS.iterdir()):
    text = parse(p)
    flag = '  <-- empty' if len(text.strip()) < 50 else ''
    print(f'{len(text):>7,} {p.name}{flag}')

  6,120 kaduna-agro-annual-report-2024.pdf
  1,689 kaduna-agro-board-deck-2025-01.pptx
  4,057 kaduna-agro-board-minutes-2024-10-17.pdf
   skipping hidden sheet: 'Draft - DO NOT CIRCULATE'
  1,050 kaduna-agro-distribution-2024.xlsx
      0 kdirs-guidance-note-4-2024-SCANNED.pdf  <-- empty
  4,286 nfsc-circular-2024-07-cybersecurity.pdf
  2,300 nfsc-circular-2025-02-amendment.pdf
  1,368 nfsc-circular-2025-02.html
    897 sahel-approved-vendors.csv
  1,298 sahel-branch-runbook.md
  6,472 sahel-employee-handbook-2023.pdf
  6,559 sahel-employee-handbook-2025.pdf
  1,385 sahel-hr-memo-2024-41-TRACKED.docx
    716 sahel-per-diem-thread.eml
  3,595 sahel-procurement-policy-v3.pdf


Fifteen documents, all readable — except the scan, which returns nothing and
says nothing about it.

Twelve questions that module 02 could not reach are now reachable. The ceiling
moves from 0.667 to 1.0. You'll measure the actual gain in notebook 6, once the
cleaning and metadata work is done.

## What's next

That `HANDLERS` dictionary and the `parse()` function around it have a name.
You've just written a general document parser — badly, in one notebook.

Notebook 2 introduces the ones other people wrote, runs them over the same
fifteen documents, and asks whether yours or theirs handles the tracked changes,
the hidden sheet and the speaker notes correctly.

The answer is not the one you'd expect.